In [1]:

!pip install -U peft transformers torchao bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from google.colab import drive
import yaml, os, torch, glob
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import classification_report
import numpy as np

# 1. Mount Drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/dataMiningProject/CSI_Project"
YAML_PATH = f"{BASE_DIR}/config.yaml"

with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['learning_rate'] = 8e-6
cfg['epochs'] = 30
cfg['rdrop_alpha'] = 1.0
cfg['scl_weight'] = 0.25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Config Loaded. Training {cfg['model_name']} for {cfg['epochs']} epochs.")

Mounted at /content/drive
 Config Loaded. Training microsoft/graphcodebert-base for 30 epochs.


In [21]:
class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Base Model
        base_encoder = AutoModel.from_pretrained(cfg['model_name'])

        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=cfg['lora_target_modules'],
            bias="none"
        )
        self.encoder = get_peft_model(base_encoder, peft_config)

        self.cwe_head = nn.ModuleDict({
            'norm': nn.LayerNorm(768),
            'fc1': nn.Linear(768, 384),
            'fc2': nn.Linear(384, cfg['num_cwe_classes'])
        })

    def get_embeddings(self, input_ids):
        return self.encoder.base_model.model.embeddings.word_embeddings(input_ids)
        if not emb.requires_grad:
            emb.requires_grad_(True)
        return emb

    def forward(self, input_ids, attention_mask, noise=None):
        if noise is not None:

            inputs_embeds = self.get_embeddings(input_ids)

            inputs_embeds = inputs_embeds + noise

            outputs = self.encoder(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        else:

            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # features (CLS token)
        features = outputs.last_hidden_state[:, 0, :]

        # Normalize
        features = torch.nn.functional.normalize(features, p=2, dim=1)

        # Classification Head
        x = self.cwe_head['norm'](features)
        x = self.cwe_head['fc1'](x)
        x = torch.relu(x)
        logits = self.cwe_head['fc2'](x)


        return {
            "logits": logits,
            "features": features,
            "hidden_states": outputs.last_hidden_state
        }

# Loss Functions
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, p=2, dim=1)
        logits = torch.matmul(features, features.T) / self.temperature
        mask = torch.eq(labels.view(-1, 1), labels.view(-1, 1).T).float().to(DEVICE)
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(labels.shape[0]).view(-1, 1).to(DEVICE),
            0
        )
        mask = mask * logits_mask
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach() # Stability trick

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)

        return -mean_log_prob_pos.mean()

class RDropLoss(nn.Module):
    def __init__(self, alpha):
        super().__init__()
        self.alpha = alpha
    def forward(self, l1, l2, target, weights):
        ce = nn.CrossEntropyLoss(weight=weights)
        kl = nn.KLDivLoss(reduction='batchmean')
        ce_loss = 0.5 * (ce(l1, target) + ce(l2, target))
        kl_loss = 0.5 * (kl(F.log_softmax(l1, dim=-1), F.softmax(l2, dim=-1)) +
                         kl(F.log_softmax(l2, dim=-1), F.softmax(l1, dim=-1)))
        return ce_loss + self.alpha * kl_loss

model = GraphCodeBERTLoRACWEModel().to(DEVICE)
scl_criterion = SupervisedContrastiveLoss(cfg['scl_temperature']).to(DEVICE)
rdrop_criterion = RDropLoss(cfg['rdrop_alpha']).to(DEVICE)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
from torch.utils.data import  WeightedRandomSampler

def get_balanced_sampler(dataset):
    labels = dataset.labels.numpy()
    class_counts = np.bincount(labels, minlength=cfg['num_cwe_classes'])
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler, class_weights


cache_path = f"{BASE_DIR}/{cfg['token_cache_file']}"
if os.path.exists(cache_path):
    cache = torch.load(cache_path, map_location='cpu')
    print(" Cache file loaded successfully.")
else:
    raise FileNotFoundError(f" Cache Path incorrect: {cache_path}")

class VulnerabilityDataset(Dataset):
    def __init__(self, cache: dict, split: str):
        indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
        self.labels = self.cwe_labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.cwe_labels[idx]
        }

train_ds = VulnerabilityDataset(cache, split="train")
val_ds = VulnerabilityDataset(cache, split="val")

counts = np.bincount(train_ds.labels.numpy(), minlength=cfg['num_cwe_classes']).astype(float)
CLASS_WEIGHTS = torch.tensor(len(train_ds) / (cfg['num_cwe_classes'] * counts), dtype=torch.float).to(DEVICE)

sampler, _ = get_balanced_sampler(train_ds)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'])

print(f" DataLoaders ready.")
print(f" Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
print(f" CLASS_WEIGHTS: {CLASS_WEIGHTS}")

 Cache file loaded successfully.
 DataLoaders ready.
 Train samples: 12994 | Val samples: 1528
 CLASS_WEIGHTS: tensor([1.6762, 1.5166, 1.1727, 2.2159, 0.5737, 1.2194, 1.7693, 0.4326],
       device='cuda:0')


In [23]:
def validate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(ids, mask)
            logits = outputs['logits']
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    return macro_f1

In [24]:
from transformers import get_cosine_schedule_with_warmup

def train_ultimate():

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg['learning_rate']), weight_decay=0.01)

    num_training_steps = len(train_loader) * cfg['epochs']
    num_warmup_steps = int(num_training_steps * cfg.get('warmup_steps', 0.1))

    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps, num_training_steps)
    scaler = torch.amp.GradScaler('cuda')


    start_epoch = 0
    best_f1 = 0.0
    weights_tensor = torch.ones(cfg['num_cwe_classes']).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=cfg.get('label_smoothing', 0.05),
        weight=weights_tensor
    )


    ckpt_path = f"{BASE_DIR}/{cfg['checkpoint_dir']}/best_model.pt"
    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
                model.load_state_dict(ckpt['model_state_dict'], strict=False)
                start_epoch = ckpt.get('epoch', -1) + 1
                best_f1 = ckpt.get('val_f1', 0.0)
                if 'class_weights' in ckpt:
                    weights_tensor = torch.tensor(ckpt['class_weights']).float().to(DEVICE)
                    criterion.weight = weights_tensor
                print(f" Resuming from Epoch {start_epoch} | Previous Best F1: {best_f1:.4f}")
            else:
                model.load_state_dict(ckpt, strict=False)
                print(" Weights loaded directly.")
        except Exception as e:
            print(f" Note: {e}. Starting fresh.")


    for epoch in range(start_epoch, cfg['epochs']):
        model.train()
        total_loss = 0

        for b in train_loader:

            ids = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbls = b['labels'].to(DEVICE) if 'labels' in b else b['cwe_label'].to(DEVICE)

            opt.zero_grad()


            with torch.amp.autocast('cuda', enabled=cfg['use_amp']):

                embeddings = model.get_embeddings(ids)


                if embeddings.is_leaf is False:

                    embeddings.retain_grad()
                else:

                    embeddings.requires_grad_(True)
                    embeddings.retain_grad()

                outputs = model(ids, mask)
                logits = outputs['logits']
                features = outputs['features']

                loss_ce = criterion(logits, lbls)
                loss_scl = scl_criterion(features, lbls)
                main_loss = loss_ce + (cfg['scl_weight'] * loss_scl)


            scaler.scale(main_loss).backward(retain_graph=True)


            if cfg.get('edat_epsilon', 0) > 0 and embeddings.grad is not None:
                with torch.amp.autocast('cuda', enabled=cfg['use_amp']):

                    noise = cfg['edat_epsilon'] * torch.sign(embeddings.grad)


                    adv_outputs = model(ids, mask, noise=noise)
                    adv_logits = adv_outputs['logits']


                    loss_adv = F.kl_div(
                        F.log_softmax(adv_logits, dim=-1),
                        F.softmax(logits.detach(), dim=-1),
                        reduction='batchmean'
                    )
                    combined_loss = cfg.get('edat_alpha', 1.0) * loss_adv

                scaler.scale(combined_loss).backward()

            scaler.step(opt)
            scaler.update()
            sched.step()
            total_loss += main_loss.item()


        f1 = validate(model, val_loader)
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | Val F1: {f1:.4f}")


        save_dict = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': f1,
            'class_weights': weights_tensor.cpu().numpy().tolist(),
            'optimizer_state_dict': opt.state_dict(),
            'cfg': cfg
        }

        torch.save(save_dict, f"{BASE_DIR}/{cfg['checkpoint_dir']}/latest_ultimate.pt")


        if f1 > best_f1:
            best_f1 = f1
            torch.save(save_dict, f"{BASE_DIR}/{cfg['checkpoint_dir']}/best_model_ultimate.pt")
            print(f" New Record Best F1 Updated: {best_f1:.4f}")


    baseline_f1 = 0.7420
    improvement = ((best_f1 - baseline_f1) / baseline_f1) * 100
    print(f"\n Training Complete! Best F1: {best_f1:.4f} | Improvement vs Baseline: {improvement:.2f}%")


train_ultimate()

 Resuming from Epoch 12 | Previous Best F1: 0.6923
Epoch 13 | Avg Loss: 0.7355 | Val F1: 0.6684
Epoch 14 | Avg Loss: 0.6970 | Val F1: 0.6812
Epoch 15 | Avg Loss: 0.6726 | Val F1: 0.6807
Epoch 16 | Avg Loss: 0.6519 | Val F1: 0.6845
Epoch 17 | Avg Loss: 0.6259 | Val F1: 0.6814
Epoch 18 | Avg Loss: 0.6122 | Val F1: 0.6834
Epoch 19 | Avg Loss: 0.5893 | Val F1: 0.6824
Epoch 20 | Avg Loss: 0.5806 | Val F1: 0.6882
Epoch 21 | Avg Loss: 0.5719 | Val F1: 0.6974
 New Record Best F1 Updated: 0.6974
Epoch 22 | Avg Loss: 0.5570 | Val F1: 0.6863
Epoch 23 | Avg Loss: 0.5553 | Val F1: 0.6976
 New Record Best F1 Updated: 0.6976
Epoch 24 | Avg Loss: 0.5513 | Val F1: 0.6925
Epoch 25 | Avg Loss: 0.5417 | Val F1: 0.6933
Epoch 26 | Avg Loss: 0.5399 | Val F1: 0.6998
 New Record Best F1 Updated: 0.6998
Epoch 27 | Avg Loss: 0.5258 | Val F1: 0.6963
Epoch 28 | Avg Loss: 0.5274 | Val F1: 0.6934
Epoch 29 | Avg Loss: 0.5202 | Val F1: 0.6935
Epoch 30 | Avg Loss: 0.5141 | Val F1: 0.7012
 New Record Best F1 Updated: 0.